# Athena emulator parity — awswrangler read paths

Fourth notebook of the `athena_emulator` parity series: exercises awswrangler's read surface against the live stack — `wr.s3.to_parquet(dataset=True)` tables registered in Glue and read back through Trino, `read_sql_query` in all three approaches (csv / ctas / managed-api) plus cache, `categories`, `chunksize`, `params` (named + qmark), and `dtype_backend`; then the read-side helpers (`read_sql_table`, `get_query_results`, `list_query_executions`, `get_query_executions`, `get_query_execution`, `wait_query`, `describe_table`, `show_create_table`, `get_query_columns_types`, `create_athena_bucket`, `get_work_group`). Boundary probes measure `unload_approach`/`unload()`, `repair_table` (MSCK), and `SHOW PARTITIONS` against the `"table$partitions"` virtual table. Each probe records a measured `PASS`/`FAIL`/`GAP`; the session matrix persists to `PARITY.md`.

Stack (`docker-compose.yml`, network `mlops_net`): `athena` (emulator) · `trino` (query engine) · `moto` (S3/Glue) · `jupyterlab` (this notebook).

In [1]:
import os

# Compose sets these for in-container runs; the localhost fallbacks let the
# same notebook run against the published ports from the host.
os.environ.setdefault("AWS_ENDPOINT_URL", "http://localhost:5000")
os.environ.setdefault("AWS_ENDPOINT_URL_ATHENA", "http://localhost:5001")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ATHENA_URL = os.environ["AWS_ENDPOINT_URL_ATHENA"]
EDGE_URL = os.environ["AWS_ENDPOINT_URL"]

print(f"athena → {ATHENA_URL}")
print(f"s3/glue → {EDGE_URL}")

athena → http://localhost:5001
s3/glue → http://localhost:5000


In [2]:
import time
import uuid
from collections.abc import Callable
from typing import TypeVar

import awswrangler as wr
import boto3
import pandas as pd
from _evidence import persist, record, render
from botocore.exceptions import ClientError

athena = boto3.client("athena")
s3 = boto3.client("s3")
glue = boto3.client("glue")

assert athena.meta.endpoint_url.rstrip("/") == ATHENA_URL

RUN_SUFFIX = uuid.uuid4().hex[:8]
# Every resource this run creates carries the nb04- prefix (nb04_ for Glue
# databases and tables, where '-' is invalid) so cleanup can discover
# leftovers — its own and any failed run's — instead of tracking a list.
ATHENA_RESOURCE_PREFIX = "nb04-"
DATABASE_PREFIX = "nb04_"
# Buckets that existed before this run: the wrangler default results bucket
# is only cleaned up when this run's create_athena_bucket call created it.
pre_existing_buckets = {info["Name"] for info in s3.list_buckets()["Buckets"]}

bucket = f"{ATHENA_RESOURCE_PREFIX}{uuid.uuid4().hex[:12]}"
database = f"{DATABASE_PREFIX}{RUN_SUFFIX}"
managed_workgroup = f"{ATHENA_RESOURCE_PREFIX}mgr-{RUN_SUFFIX}"
people_table = f"{DATABASE_PREFIX}people_{RUN_SUFFIX}"
sales_table = f"{DATABASE_PREFIX}sales_{RUN_SUFFIX}"
results_prefix = f"s3://{bucket}/results/"

s3.create_bucket(Bucket=bucket)
glue.create_database(DatabaseInput={"Name": database})
# Managed results: GetQueryExecution reports an empty ResultConfiguration and
# wrangler falls back to inline GetQueryResults reads (the api path).
athena.create_work_group(
    Name=managed_workgroup,
    Configuration={"ManagedQueryResultsConfiguration": {"Enabled": True}},
)

{'ResponseMetadata': {'RequestId': '53393554-0a1f-4981-9b44-f28c0517793a',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Tue, 06 Oct 2026 21:58:21 GMT',
   'server': 'uvicorn',
   'content-type': 'application/x-amz-json-1.1',
   'x-amzn-requestid': '53393554-0a1f-4981-9b44-f28c0517793a',
   'content-length': '2'},
  'RetryAttempts': 0}}

## Helpers

`probe_outcome` returns a probe call's value or the exception it raised —
either is a measured outcome, so boundary probes never work around a failure.
`error_detail` flattens a `ClientError` (or any exception) to one matrix
line. Cleanup helpers are prefix-discovered, matching NB-02/03.

In [3]:
T = TypeVar("T")


def probe_outcome(call: Callable[[], T]) -> T | Exception:
    """``call()``'s value, or the exception it raised — both are evidence."""
    try:
        return call()
    except Exception as exc:  # a measured probe records any failure
        return exc


def error_detail(error: Exception) -> str:
    """One matrix line: wire Code+status for ClientError, else type+text."""
    if isinstance(error, ClientError):
        code = error.response["Error"]["Code"]
        status = error.response["ResponseMetadata"]["HTTPStatusCode"]
        message = error.response["Error"].get("Message", "")
        return f"{code} ({status}): {message.splitlines()[0][:80]}"
    first = str(error).splitlines()[0] if str(error) else ""
    return f"{type(error).__name__}: {first[:80]}"


def object_keys_under(prefix: str) -> list[str]:
    """Object keys under ``prefix`` in the throwaway bucket."""
    contents = s3.list_objects_v2(Bucket=bucket, Prefix=prefix).get(
        "Contents", []
    )
    return [obj["Key"] for obj in contents]


def run_workgroup_names() -> list[str]:
    """Workgroup names carrying this run's (or a failed run's) prefix."""
    return [
        summary["Name"]
        for summary in athena.list_work_groups()["WorkGroups"]
        if summary["Name"].startswith(ATHENA_RESOURCE_PREFIX)
    ]


def delete_run_buckets() -> None:
    """Empty and delete run-prefixed buckets plus this run's default."""
    for bucket_info in s3.list_buckets()["Buckets"]:
        name = bucket_info["Name"]
        # create_athena_bucket() names its bucket after the moto account, so
        # it never carries the run prefix — remove it only when this run
        # created it (it was absent from the setup-time listing).
        is_default_results = (
            name.startswith("aws-athena-query-results-")
            and name not in pre_existing_buckets
        )
        if not (name.startswith(ATHENA_RESOURCE_PREFIX) or is_default_results):
            continue
        objects = s3.list_objects_v2(Bucket=name).get("Contents", [])
        if objects:
            s3.delete_objects(
                Bucket=name,
                Delete={"Objects": [{"Key": obj["Key"]} for obj in objects]},
            )
        s3.delete_bucket(Bucket=name)


def delete_run_databases() -> None:
    """Delete tables, then every Glue database carrying the run prefix."""
    for database_info in glue.get_databases()["DatabaseList"]:
        database_name = database_info["Name"]
        if not database_name.startswith(DATABASE_PREFIX):
            continue
        tables = glue.get_tables(DatabaseName=database_name)["TableList"]
        for table in tables:
            glue.delete_table(DatabaseName=database_name, Name=table["Name"])
        glue.delete_database(Name=database_name)


def cleanup_run_resources() -> None:
    """Remove everything this run (or a failed earlier run) left behind."""
    for name in run_workgroup_names():
        athena.delete_work_group(WorkGroup=name)
    delete_run_buckets()
    delete_run_databases()

## Dataset — `wr.s3.to_parquet(dataset=True)` + Glue registration

`dataset=True` writes parquet under the dataset prefix and registers the
table (plus `partition_cols` partitions) in the Glue catalog — so the read
probes exercise Trino's Glue-metastore resolution, not a Trino-created
table.

In [4]:
people = pd.DataFrame(
    {
        "id": [1, 2, 3],
        "name": ["alpha", "beta", "gamma"],
        "score": [9.5, 8.25, 7.0],
    }
)
wr.s3.to_parquet(
    df=people,
    path=f"s3://{bucket}/datasets/people/",
    dataset=True,
    database=database,
    table=people_table,
)
people_columns = wr.catalog.table(database=database, table=people_table)
assert people_columns["Column Name"].tolist() == ["id", "name", "score"]

sales = pd.DataFrame(
    {
        "quantity": [3, 7, 5],
        "region": ["US", "EU", "US"],
        "amount": [10.5, 3.14, 3.14],
    }
)
wr.s3.to_parquet(
    df=sales,
    path=f"s3://{bucket}/datasets/sales/",
    dataset=True,
    database=database,
    table=sales_table,
    partition_cols=["region"],
)
sales_partitions = glue.get_partitions(
    DatabaseName=database, TableName=sales_table
)["Partitions"]
partition_values = sorted(p["Values"][0] for p in sales_partitions)
assert partition_values == ["EU", "US"]

record(
    "wr.s3.to_parquet(dataset=True) registers Glue tables incl. partitions",
    "PASS",
    f"{people_table}: {len(people_columns)} columns registered; "
    f"{sales_table}: {len(sales_partitions)} partitions {partition_values}",
)

Evidence(feature='wr.s3.to_parquet(dataset=True) registers Glue tables incl. partitions', status='PASS', detail="nb04_people_e74ede2c: 3 columns registered; nb04_sales_e74ede2c: 2 partitions ['EU', 'US']", latency_ms=None)

## `read_sql_query` — csv / ctas / api approaches

- csv (`ctas_approach=False`): read the `{qid}.csv` artifact back from S3.
- ctas (default): wrap the query in a temp CTAS, then read the parquet files
  the `-manifest.csv` lists.
- api: a managed-results workgroup reports no `OutputLocation`, so rows come
  back inline via `GetQueryResults`.

In [5]:
feature = "read_sql_query csv path reads a wrangler-registered table"
started = time.monotonic()
csv_outcome = probe_outcome(
    lambda: wr.athena.read_sql_query(
        sql=f'SELECT * FROM "{database}"."{people_table}" ORDER BY id',
        database=database,
        ctas_approach=False,
        s3_output=results_prefix,
    )
)
if isinstance(csv_outcome, pd.DataFrame):
    assert csv_outcome["name"].tolist() == ["alpha", "beta", "gamma"]
    csv_qid = csv_outcome.query_metadata["QueryExecutionId"]
    record(
        feature,
        "PASS",
        f"{csv_qid} shape={csv_outcome.shape} — Trino resolved the "
        "Glue-registered parquet table",
        (time.monotonic() - started) * 1000,
    )
if not isinstance(csv_outcome, pd.DataFrame):
    csv_qid = None
    record(feature, "GAP", error_detail(csv_outcome))

In [6]:
started = time.monotonic()
ctas_frame = wr.athena.read_sql_query(
    sql="SELECT 1 AS one, 'alpha' AS label",
    database=database,
    s3_output=results_prefix,
)
ctas_qid = ctas_frame.query_metadata["QueryExecutionId"]
ctas_execution = athena.get_query_execution(QueryExecutionId=ctas_qid)[
    "QueryExecution"
]
assert ctas_frame.loc[0, "one"] == 1
assert ctas_execution["StatementType"] == "DDL"
assert ctas_execution["SubstatementType"] == "CREATE_TABLE_AS_SELECT"
ctas_manifest = ctas_execution["Statistics"]["DataManifestLocation"]
assert ctas_manifest.endswith("-manifest.csv")
record(
    "read_sql_query ctas_approach (temp CTAS → manifest → parquet read)",
    "PASS",
    f"{ctas_qid} rows={len(ctas_frame)}; "
    f"manifest={ctas_manifest.split('/')[-1]}",
    (time.monotonic() - started) * 1000,
)

Evidence(feature='read_sql_query ctas_approach (temp CTAS → manifest → parquet read)', status='PASS', detail='0c4af262-6f39-4e0c-8dc0-9e9dc247544a rows=1; manifest=0c4af262-6f39-4e0c-8dc0-9e9dc247544a-manifest.csv', latency_ms=1313.1048600043869)

In [7]:
started = time.monotonic()
api_frame = wr.athena.read_sql_query(
    sql="SELECT 2 AS two, 'beta' AS label",
    database=database,
    ctas_approach=False,
    workgroup=managed_workgroup,
)
api_qid = api_frame.query_metadata["QueryExecutionId"]
managed_execution = athena.get_query_execution(QueryExecutionId=api_qid)[
    "QueryExecution"
]
assert api_frame.loc[0, "two"] == 2
# Managed results report an empty ResultConfiguration — the signal wrangler
# uses to fall back to inline GetQueryResults.
assert managed_execution["ResultConfiguration"] == {}
record(
    "read_sql_query api path (managed workgroup → inline GetQueryResults)",
    "PASS",
    f"{api_qid} ResultConfiguration={{}}; {len(api_frame)} row(s) served "
    "inline",
    (time.monotonic() - started) * 1000,
)

Evidence(feature='read_sql_query api path (managed workgroup → inline GetQueryResults)', status='PASS', detail='6e017fe5-8e9e-481e-b1a0-4763c3254826 ResultConfiguration={}; 1 row(s) served inline', latency_ms=228.5157629958121)

## Cache and fetch options

`athena_cache_settings` reuses a recent identical execution;
`categories`, `chunksize`, and `dtype_backend` shape the returned frames;
`params` formats client-side (`named`) or binds server-side (`qmark` →
`ExecutionParameters`).

In [8]:
cache_sql = "SELECT 7 AS seven, 'cached' AS label"
cache_settings = {"max_cache_seconds": 90}
cache_first = wr.athena.read_sql_query(
    sql=cache_sql,
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
    athena_cache_settings=cache_settings,
)
cache_first_id = cache_first.query_metadata["QueryExecutionId"]
cache_second = wr.athena.read_sql_query(
    sql=cache_sql,
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
    athena_cache_settings=cache_settings,
)
cache_second_id = cache_second.query_metadata["QueryExecutionId"]
# A hit re-reads the first execution's csv instead of running again.
assert cache_second_id == cache_first_id
record(
    "athena_cache_settings cache hit reuses the first execution",
    "PASS",
    f"both reads → {cache_first_id}; list+batch_get supplied the hit",
)

Evidence(feature='athena_cache_settings cache hit reuses the first execution', status='PASS', detail='both reads → 9cefe399-e84d-4801-bd21-aa70dd59834b; list+batch_get supplied the hit', latency_ms=None)

In [9]:
feature = "read_sql_query categories → pandas Categorical"
cat_outcome = probe_outcome(
    lambda: wr.athena.read_sql_query(
        sql=f'SELECT region, quantity FROM "{database}"."{sales_table}"',
        database=database,
        ctas_approach=False,
        s3_output=results_prefix,
        categories=["region"],
    )
)
if isinstance(cat_outcome, pd.DataFrame):
    assert str(cat_outcome["region"].dtype) == "category"
    record(feature, "PASS", f"region dtype={cat_outcome['region'].dtype}")
if not isinstance(cat_outcome, pd.DataFrame):
    record(feature, "GAP", error_detail(cat_outcome))

feature = "read_sql_query chunksize=2 → iterator of row-bounded frames"
chunks_outcome = probe_outcome(
    lambda: list(
        wr.athena.read_sql_query(
            sql=f'SELECT * FROM "{database}"."{people_table}" ORDER BY id',
            database=database,
            ctas_approach=False,
            s3_output=results_prefix,
            chunksize=2,
        )
    )
)
if isinstance(chunks_outcome, list):
    chunk_sizes = [len(chunk) for chunk in chunks_outcome]
    assert chunk_sizes == [2, 1]
    record(feature, "PASS", f"chunk row counts={chunk_sizes}")
if not isinstance(chunks_outcome, list):
    record(feature, "GAP", error_detail(chunks_outcome))

feature = 'read_sql_query dtype_backend="pyarrow"'
pyarrow_outcome = probe_outcome(
    lambda: wr.athena.read_sql_query(
        sql="SELECT 1 AS one, 1.5 AS two",
        database=database,
        ctas_approach=False,
        s3_output=results_prefix,
        dtype_backend="pyarrow",
    )
)
if isinstance(pyarrow_outcome, pd.DataFrame):
    dtypes = [str(dtype) for dtype in pyarrow_outcome.dtypes]
    assert all(dtype.endswith("[pyarrow]") for dtype in dtypes)
    record(feature, "PASS", f"dtypes={dtypes}")
if not isinstance(pyarrow_outcome, pd.DataFrame):
    record(feature, "GAP", error_detail(pyarrow_outcome))

In [10]:
feature = "read_sql_query params (named → client-side format)"
named_outcome = probe_outcome(
    lambda: wr.athena.read_sql_query(
        sql=(
            f'SELECT quantity FROM "{database}"."{sales_table}" '
            "WHERE region = :region"
        ),
        database=database,
        ctas_approach=False,
        s3_output=results_prefix,
        params={"region": "EU"},
    )
)
if isinstance(named_outcome, pd.DataFrame):
    assert named_outcome["quantity"].tolist() == [7]
    record(feature, "PASS", "WHERE region = 'EU' pruned read → 1 row")
if not isinstance(named_outcome, pd.DataFrame):
    record(feature, "GAP", error_detail(named_outcome))

feature = "read_sql_query params (qmark → server-side ExecutionParameters)"
qmark_outcome = probe_outcome(
    lambda: wr.athena.read_sql_query(
        sql=(
            f'SELECT quantity FROM "{database}"."{sales_table}" '
            "WHERE region = ?"
        ),
        database=database,
        ctas_approach=False,
        s3_output=results_prefix,
        params=["EU"],
        paramstyle="qmark",
    )
)
if isinstance(qmark_outcome, pd.DataFrame):
    assert qmark_outcome["quantity"].tolist() == [7]
    qmark_qid = qmark_outcome.query_metadata["QueryExecutionId"]
    echoed = athena.get_query_execution(QueryExecutionId=qmark_qid)[
        "QueryExecution"
    ].get("ExecutionParameters")
    record(
        feature,
        "PASS",
        f"{qmark_qid} bound server-side → 1 row; "
        f"ExecutionParameters echoed={echoed}",
    )
if not isinstance(qmark_outcome, pd.DataFrame):
    record(feature, "GAP", error_detail(qmark_outcome))

## Table and result helpers

`read_sql_table` wraps a `SELECT *` in `read_sql_query`; `get_query_results`
re-reads a finished execution — the csv artifact for a DML `SELECT`, the
manifest-listed parquet for a CTAS execution.

In [11]:
feature = "read_sql_table reads the full registered table"
table_outcome = probe_outcome(
    lambda: wr.athena.read_sql_table(
        table=people_table,
        database=database,
        ctas_approach=False,
        s3_output=results_prefix,
    )
)
if isinstance(table_outcome, pd.DataFrame):
    assert sorted(table_outcome["name"].tolist()) == [
        "alpha",
        "beta",
        "gamma",
    ]
    record(feature, "PASS", f"shape={table_outcome.shape}")
if not isinstance(table_outcome, pd.DataFrame):
    record(feature, "GAP", error_detail(table_outcome))

In [12]:
feature = "get_query_results fetches the csv artifact of a DML SELECT"
csv_results = probe_outcome(
    lambda: wr.athena.get_query_results(query_execution_id=cache_first_id)
)
if isinstance(csv_results, pd.DataFrame):
    assert csv_results.loc[0, "seven"] == 7
    record(
        feature, "PASS", f"{cache_first_id[:8]}… → shape={csv_results.shape}"
    )
if not isinstance(csv_results, pd.DataFrame):
    record(feature, "GAP", error_detail(csv_results))

feature = "get_query_results fetches manifest parquet of a CTAS execution"
ctas_results = probe_outcome(
    lambda: wr.athena.get_query_results(query_execution_id=ctas_qid)
)
if isinstance(ctas_results, pd.DataFrame):
    assert ctas_results.loc[0, "one"] == 1
    record(feature, "PASS", f"{ctas_qid[:8]}… → shape={ctas_results.shape}")
if not isinstance(ctas_results, pd.DataFrame):
    record(feature, "GAP", error_detail(ctas_results))

## Execution metadata helpers

Wrangler's metadata calls map onto `ListQueryExecutions`,
`BatchGetQueryExecution`, and `GetQueryExecution` (`wait_query` polls the
last) — the same wire surface the boto3 lifecycle notebook measured.

In [13]:
feature = "list_query_executions returns the workgroup's execution ids"
listed = probe_outcome(
    lambda: wr.athena.list_query_executions(workgroup="primary")
)
if isinstance(listed, list):
    assert cache_first_id in listed
    record(feature, "PASS", f"{len(listed)} id(s) incl. {cache_first_id[:8]}…")
if not isinstance(listed, list):
    record(feature, "GAP", error_detail(listed))

feature = "get_query_executions batch_gets into frames (+ unprocessed ids)"
executions = probe_outcome(
    lambda: wr.athena.get_query_executions(
        [cache_first_id, ctas_qid, "00000000-0000-0000-0000-000000000000"],
        return_unprocessed=True,
    )
)
if isinstance(executions, tuple):
    frame, unprocessed = executions
    assert set(frame["QueryExecutionId"]) == {cache_first_id, ctas_qid}
    assert len(unprocessed) == 1
    record(
        feature,
        "PASS",
        f"{len(frame)} execution rows + {len(unprocessed)} unprocessed",
    )
if not isinstance(executions, tuple):
    record(feature, "GAP", error_detail(executions))

feature = "get_query_execution returns the execution member"
fetched = probe_outcome(lambda: wr.athena.get_query_execution(cache_first_id))
if isinstance(fetched, dict):
    assert fetched["Status"]["State"] == "SUCCEEDED"
    record(
        feature,
        "PASS",
        f"{cache_first_id[:8]}… Status.State={fetched['Status']['State']}",
    )
if not isinstance(fetched, dict):
    record(feature, "GAP", error_detail(fetched))

feature = "wait_query returns on a terminal execution"
waited = probe_outcome(lambda: wr.athena.wait_query(cache_first_id))
if isinstance(waited, dict):
    assert waited["Status"]["State"] == "SUCCEEDED"
    record(feature, "PASS", f"{cache_first_id[:8]}… returned SUCCEEDED")
if not isinstance(waited, dict):
    record(feature, "GAP", error_detail(waited))

## Schema helpers — `DESCRIBE` / `SHOW CREATE TABLE` / column types

Wrangler spells these `DESCRIBE \`table\`;` and `SHOW CREATE TABLE
\`table\`;` — backtick-quoted with a statement terminator. Whether that
survives the Trino dialect is measured, not assumed.

In [14]:
feature = "describe_table returns the table's column listing"
described = probe_outcome(
    lambda: wr.athena.describe_table(
        table=people_table, database=database, s3_output=results_prefix
    )
)
if isinstance(described, pd.DataFrame):
    record(feature, "PASS", f"columns={described.iloc[:, 0].tolist()}")
if not isinstance(described, pd.DataFrame):
    record(feature, "GAP", error_detail(described))

feature = "show_create_table returns the CREATE TABLE statement"
shown = probe_outcome(
    lambda: wr.athena.show_create_table(
        table=people_table, database=database, s3_output=results_prefix
    )
)
if isinstance(shown, str):
    record(feature, "PASS", shown.splitlines()[0][:80])
if not isinstance(shown, str):
    record(feature, "GAP", error_detail(shown))

In [15]:
feature = "get_query_columns_types maps ColumnInfo Name/Type"
types_outcome = probe_outcome(
    lambda: wr.athena.get_query_columns_types(cache_first_id)
)
if isinstance(types_outcome, dict):
    record(feature, "PASS", f"{types_outcome}")
if not isinstance(types_outcome, dict):
    record(feature, "GAP", error_detail(types_outcome))

feature = "get_query_columns_types formats decimal precision/scale"
decimal_frame = wr.athena.read_sql_query(
    sql="SELECT CAST(12.34 AS DECIMAL(6, 2)) AS amount",
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
)
decimal_types = probe_outcome(
    lambda: wr.athena.get_query_columns_types(
        decimal_frame.query_metadata["QueryExecutionId"]
    )
)
if isinstance(decimal_types, dict):
    record(feature, "PASS", f"{decimal_types}")
if not isinstance(decimal_types, dict):
    record(feature, "GAP", error_detail(decimal_types))

## Bucket and workgroup helpers

`create_athena_bucket` derives `aws-athena-query-results-{account}-{region}`
through STS `GetCallerIdentity` (moto answers `123456789012`) and creates it
on moto S3.

In [16]:
feature = "create_athena_bucket creates the default results bucket"
default_path = probe_outcome(lambda: wr.athena.create_athena_bucket())
if isinstance(default_path, str):
    default_bucket = default_path.removeprefix("s3://").rstrip("/")
    s3.head_bucket(Bucket=default_bucket)
    record(feature, "PASS", f"{default_path} exists on moto")
if not isinstance(default_path, str):
    record(feature, "GAP", error_detail(default_path))

feature = "get_work_group returns primary's configuration"
primary_wg = probe_outcome(lambda: wr.athena.get_work_group("primary"))
if isinstance(primary_wg, dict):
    primary_member = primary_wg["WorkGroup"]
    record(
        feature,
        "PASS",
        f"state={primary_member['State']} "
        f"enforce={primary_member['Configuration']['EnforceWorkGroupConfiguration']}",
    )
if not isinstance(primary_wg, dict):
    record(feature, "GAP", error_detail(primary_wg))

## Boundary probes — `unload_approach` / `unload()`

UNLOAD is Athena syntax for writing query results as parquet/text to a path.
Trino 483 has no `UNLOAD` statement — the probes measure the emulator's
actual response instead of assuming a mapping exists.

In [17]:
feature = "read_sql_query unload_approach=True (UNLOAD → parquet read)"
# s3_output is the UNLOAD TO path verbatim (awswrangler/athena/_read.py:780-790);
# a dedicated subdir is required — Athena rejects UNLOAD into a non-empty
# location, and results_prefix already holds this run's earlier probe files.
unload_read = probe_outcome(
    lambda: wr.athena.read_sql_query(
        sql="SELECT 1 AS one, 'alpha' AS label",
        database=database,
        ctas_approach=False,
        unload_approach=True,
        s3_output=f"{results_prefix}unload/",
        unload_parameters={"file_format": "PARQUET"},
    )
)
if isinstance(unload_read, pd.DataFrame):
    record(feature, "PASS", f"shape={unload_read.shape}")
if not isinstance(unload_read, pd.DataFrame):
    record(feature, "GAP", error_detail(unload_read))

feature = "wr.athena.unload() writes a result set to S3 via UNLOAD"
unloaded = probe_outcome(
    lambda: wr.athena.unload(
        sql="SELECT * FROM (VALUES (1,'a'),(2,'b')) t(n,s)",
        path=f"s3://{bucket}/unload/",
        database=database,
    )
)
if isinstance(unloaded, Exception):
    record(feature, "GAP", error_detail(unloaded))
if not isinstance(unloaded, Exception):
    record(feature, "PASS", f"unloaded under s3://{bucket}/unload/")

## `repair_table` (MSCK) and `SHOW PARTITIONS` vs `"table$partitions"`

A stray parquet file lands under a new `region=XX/` prefix — unregistered in
Glue until `repair_table` runs. `repair_table` submits `MSCK REPAIR TABLE`
(Hive syntax, not Trino); `SHOW PARTITIONS <t>` is Athena's spelling while
Trino's workaround is the `"<t>$partitions"` virtual table. All three are
measured.

In [18]:
extra = pd.DataFrame({"quantity": [9], "amount": [1.0]})
wr.s3.to_parquet(
    df=extra,
    path=f"s3://{bucket}/datasets/sales/region=XX/extra.parquet",
)

feature = "repair_table (MSCK REPAIR TABLE) discovers new partitions"
repaired = probe_outcome(
    lambda: wr.athena.repair_table(
        table=sales_table, database=database, s3_output=results_prefix
    )
)
if isinstance(repaired, str):
    record(feature, "PASS", f"final state {repaired}")
if not isinstance(repaired, str):
    record(feature, "GAP", error_detail(repaired))

feature = "SHOW PARTITIONS <table> (Athena spelling)"
shown_partitions = probe_outcome(
    lambda: wr.athena.read_sql_query(
        sql=f"SHOW PARTITIONS {sales_table}",
        database=database,
        ctas_approach=False,
        s3_output=results_prefix,
    )
)
if isinstance(shown_partitions, pd.DataFrame):
    record(feature, "PASS", f"rows={len(shown_partitions)}")
if not isinstance(shown_partitions, pd.DataFrame):
    record(feature, "GAP", error_detail(shown_partitions))

feature = '"<table>$partitions" virtual table lists Glue partitions'
partition_rows = probe_outcome(
    lambda: wr.athena.read_sql_query(
        sql=f'SELECT region FROM "{database}"."{sales_table}$partitions"',
        database=database,
        ctas_approach=False,
        s3_output=results_prefix,
    )
)
if isinstance(partition_rows, pd.DataFrame):
    listed_regions = sorted(partition_rows["region"].tolist())
    # XX was registered by the MSCK repair above — Glue's view now lists
    # all three partitions.
    assert listed_regions == ["EU", "US", "XX"]
    record(feature, "PASS", f"Glue partitions={listed_regions}")
if not isinstance(partition_rows, pd.DataFrame):
    record(feature, "GAP", error_detail(partition_rows))

## Session matrix

In [19]:
print(render())
persist("04_wrangler_read_paths")

| feature | status | detail | latency_ms |
|---|---|---|---|
| wr.s3.to_parquet(dataset=True) registers Glue tables incl. partitions | PASS | nb04_people_e74ede2c: 3 columns registered; nb04_sales_e74ede2c: 2 partitions ['EU', 'US'] |  |
| read_sql_query csv path reads a wrangler-registered table | PASS | 5eabbb46-e7de-4709-9c79-7ca392a2ba4e shape=(3, 3) — Trino resolved the Glue-registered parquet table | 1340.56 |
| read_sql_query ctas_approach (temp CTAS → manifest → parquet read) | PASS | 0c4af262-6f39-4e0c-8dc0-9e9dc247544a rows=1; manifest=0c4af262-6f39-4e0c-8dc0-9e9dc247544a-manifest.csv | 1313.1 |
| read_sql_query api path (managed workgroup → inline GetQueryResults) | PASS | 6e017fe5-8e9e-481e-b1a0-4763c3254826 ResultConfiguration={}; 1 row(s) served inline | 228.516 |
| athena_cache_settings cache hit reuses the first execution | PASS | both reads → 9cefe399-e84d-4801-bd21-aa70dd59834b; list+batch_get supplied the hit |  |
| read_sql_query categories → pandas Categorical | PA

PosixPath('/home/gabriel-menezes/Documents/repos/mlops/projects/athena_emulator/parity/04_wrangler_read_paths.md')

In [20]:
cleanup_run_resources()
print(
    "cleaned: "
    f"workgroups={run_workgroup_names()} "
    "nb04 buckets+glue databases (incl. tables) and this run's default "
    "results bucket removed"
)

cleaned: workgroups=[] nb04 buckets+glue databases (incl. tables) and this run's default results bucket removed
